In [1]:
!pip install detoxify -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np
from detoxify import Detoxify

/usr/local/python/3.12.1/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
df = pd.read_csv("../Outputs/merged_text_905.csv")
print(f"Loaded: {df.shape[0]} rows")
print(f"Posts with usable text: {df['has_text'].sum()}")

Loaded: 905 rows
Posts with usable text: 797


In [5]:
# 'original' = 7 subtypes; 'unbiased' = bias-adjusted version
model = Detoxify('original')
print("Detoxify loaded.")

# Quick test
print(model.predict("I'm so excited for the Super Bowl!"))

Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.1-alpha/toxic_original-c1212f89.ckpt" to /home/codespace/.cache/torch/hub/checkpoints/toxic_original-c1212f89.ckpt


100%|██████████| 418M/418M [00:04<00:00, 88.9MB/s]
Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14788.88it/s]


Detoxify loaded.
{'toxicity': np.float32(0.00069633283), 'severe_toxicity': np.float32(0.000121302684), 'obscene': np.float32(0.00018673998), 'threat': np.float32(0.00012278107), 'insult': np.float32(0.00016852368), 'identity_attack': np.float32(0.00014438565)}


In [6]:
texts = df.loc[df["has_text"], "text_for_analysis"].tolist()
texts = [t[:2000] for t in texts]

print(f"Running Detoxify on {len(texts)} posts...")

# Run in batches to avoid memory issues
batch_size = 16
all_results = []
for i in range(0, len(texts), batch_size):
    batch = texts[i:i+batch_size]
    batch_results = model.predict(batch)
    # batch_results is a dict of lists; convert to list of dicts
    for j in range(len(batch)):
        all_results.append({k: v[j] for k, v in batch_results.items()})
    if i % 80 == 0:
        print(f"  Processed {i+len(batch)} / {len(texts)}")

print(f"Done. Got {len(all_results)} results")

Running Detoxify on 797 posts...
  Processed 16 / 797
  Processed 96 / 797
  Processed 176 / 797
  Processed 256 / 797
  Processed 336 / 797
  Processed 416 / 797
  Processed 496 / 797
  Processed 576 / 797
  Processed 656 / 797
  Processed 736 / 797
Done. Got 797 results


In [9]:
tox_labels = ["toxicity", "severe_toxicity", "obscene", "identity_attack", "insult", "threat"]
tox_cols = [f"tox_{c}" for c in tox_labels]

# Drop existing tox columns if they exist (safe re-run)
df = df.drop(columns=[c for c in tox_cols + ["dominant_toxicity", "dominant_toxicity_score", "is_toxic"] if c in df.columns])

# Build score dataframe
tox_df = pd.DataFrame(all_results)[tox_labels]
tox_df.columns = tox_cols
tox_df.index = df.index[df["has_text"]]
df = df.join(tox_df)

# Apply threshold: only consider posts toxic if any score crosses 0.5
THRESHOLD = 0.5
df["is_toxic"] = None
df["dominant_toxicity"] = None
df["dominant_toxicity_score"] = None

mask = df["has_text"]
df.loc[mask, "dominant_toxicity_score"] = df.loc[mask, tox_cols].max(axis=1)
df.loc[mask, "is_toxic"] = (df.loc[mask, "dominant_toxicity_score"] > THRESHOLD)
# Only assign dominant_toxicity label if the post is actually toxic
toxic_mask = mask & (df["dominant_toxicity_score"] > THRESHOLD)
df.loc[toxic_mask, "dominant_toxicity"] = (
    df.loc[toxic_mask, tox_cols].idxmax(axis=1).str.replace("tox_", "")
)
# Non-toxic posts get labeled "not_toxic"
non_toxic_mask = mask & (df["dominant_toxicity_score"] <= THRESHOLD)
df.loc[non_toxic_mask, "dominant_toxicity"] = "not_toxic"

print(df[["student_id", "text_source", "dominant_toxicity", "dominant_toxicity_score"]].head())

  student_id         text_source dominant_toxicity dominant_toxicity_score
0        1_A  caption+transcript         not_toxic                0.000657
1        1_A  caption+transcript         not_toxic                0.003382
2        1_A        caption_only         not_toxic                0.001024
3        2_A        caption_only         not_toxic                0.000685
4        2_A        caption_only         not_toxic                0.001143


In [10]:
print("=== Overall dominant toxicity counts ===")
print(df["dominant_toxicity"].value_counts(dropna=False))
print()
print(f"=== Toxic posts (score > {THRESHOLD}) ===")
toxic_only = df[df["is_toxic"] == True]
print(f"Total toxic: {len(toxic_only)} / {df['has_text'].sum()} usable posts")
print()
print("=== Mean scores across ALL usable posts ===")
print(df[tox_cols].mean().sort_values(ascending=False))
print()
print("=== Mean scores within toxic posts only ===")
if len(toxic_only) > 0:
    print(toxic_only[tox_cols].mean().sort_values(ascending=False))

=== Overall dominant toxicity counts ===
dominant_toxicity
not_toxic    776
None         108
toxicity      20
obscene        1
Name: count, dtype: int64

=== Toxic posts (score > 0.5) ===
Total toxic: 21 / 797 usable posts

=== Mean scores across ALL usable posts ===
tox_toxicity           0.042086
tox_obscene            0.014467
tox_insult             0.008048
tox_identity_attack    0.001696
tox_threat             0.001466
tox_severe_toxicity    0.000776
dtype: float64

=== Mean scores within toxic posts only ===
tox_toxicity           0.763103
tox_obscene            0.481298
tox_insult             0.248082
tox_identity_attack    0.043528
tox_threat             0.033087
tox_severe_toxicity    0.023493
dtype: float64


In [11]:
print("=== Toxic posts by text source ===")
print(pd.crosstab(df["text_source"], df["is_toxic"]))
print()
print("=== Toxicity confidence by text source ===")
print(df.groupby("text_source")["dominant_toxicity_score"].agg(["mean", "median", "max", "count"]))

=== Toxic posts by text source ===
is_toxic            False  True 
text_source                     
caption+transcript    129     14
caption_only          646      6
transcript_only         1      1

=== Toxicity confidence by text source ===
                        mean    median       max  count
text_source                                            
caption+transcript  0.118238  0.011214  0.994399    143
caption_only        0.024924  0.001254  0.934938    652
none                     NaN       NaN       NaN      0
transcript_only     0.252923  0.252923  0.501576      2


In [12]:
toxic_only = df[df["is_toxic"] == True].sort_values("dominant_toxicity_score", ascending=False)
print(f"=== Top {min(15, len(toxic_only))} toxic posts ===")
for _, r in toxic_only.head(15).iterrows():
    text_preview = r['text_for_analysis'][:200].replace('\n', ' | ')
    print(f"\n  [{r['dominant_toxicity_score']:.2f}] [{r['dominant_toxicity']}] [{r['student_id']}] [{r['text_source']}]")
    print(f"  {text_preview}") 

=== Top 15 toxic posts ===

  [0.99] [toxicity] [12_A] [caption+transcript]
  GYMSKIN doubled his AURA after they didnt BURN THE BEAN ???? #gymskin |  | These are... Too groovy. Look at these. They didn't fucking burn the bean. Look at these fucking...

  [0.98] [toxicity] [6_A] [caption+transcript]
  dirty blonde core #dirtyblonde #blondehair #naturalhair |  | People die for this, people lie for this, people suck and fuck some guy for this, | pay the toll for this, sell their soul for this, play my part

  [0.93] [toxicity] [10_A] [caption_only]
  "pulling up to talk shit with my mom and realizing she's not in the mood to take my side" "Pivot turn exit"

  [0.91] [toxicity] [4_A] [caption+transcript]
  The best tour guide #playadelcarmen #cenotes #cenotesmexico |  | You try the no flash here. The dark part. Oh, I don't know about that. Are you going to turn it off? A couple seconds. Okay, see what it lo

  [0.90] [toxicity] [1_A] [caption+transcript]
  Look at what the last cat has d

In [13]:
output_path = "../Outputs/toxicity_detoxify_905.csv"
df.to_csv(output_path, index=False)
print(f"Saved: {output_path}")

Saved: ../Outputs/toxicity_detoxify_905.csv


# Toxicity Model 1: Detoxify — Conclusion

**Model:** `unitary/toxic-bert` (Detoxify "original")
**Output labels:** 6 toxicity subtypes — toxicity, severe_toxicity, obscene, identity_attack, insult, threat
**Threshold applied:** 0.5 (posts below are labeled "not_toxic")
**Dataset:** 905 posts from 12 students, 797 had usable text

## What we did in plain terms

We ran every post through Detoxify, which gives a score from 0 to 1 for each of 6 toxicity categories. This time we set a 0.5 threshold so posts that score below it are just labeled "not_toxic" — fixing the issue from the earlier run where every post was forced into a category like "insult."

## What we found

Out of 797 usable posts, only 21 (2.6%) crossed the 0.5 threshold and were labeled toxic:
- 20 posts tagged as general "toxicity"
- 1 post tagged as "obscene"
- 776 posts labeled "not_toxic"

For comparison, students themselves flagged 10 posts as uncivil in Q8. The model finds about 2x more than students do.

## The transcript effect

Posts with transcripts are 10x more likely to be flagged toxic than caption-only posts:
- Caption only: 6 toxic out of 652 (0.9%)
- Caption + transcript: 14 toxic out of 143 (9.8%)

This makes sense: video transcripts capture audio content (including profanity) that the caption doesn't show. Students often paste a clean caption for a video with explicit audio.

## The bigger problem: profanity ≠ toxicity

When we looked at the actual posts flagged as toxic, almost none of them are genuinely harmful:

- A gym influencer saying "they didn't fucking burn the bean" — casual fitness content
- A song with the lyric "suck and fuck some guy" — music with explicit lyrics
- A cat video where someone says "What the f***?" reacting to a mess — household swearing
- A FaceTime with a first customer — completely innocent but flagged at 71% toxic
- A book content reference "Off with his head!" — literary quote

The model is detecting profanity, not actual toxicity. It cannot tell the difference between casual swearing in entertainment content and genuine harassment or hate speech.

It also failed on two posts whose transcripts were copy-paste errors in the dataset (the Trump tariffs post and "THE DRAMA April 3rd" both have a wedding dialogue transcript attached by mistake).

## Comparing to Q8 student self-reports

Students flagged 10 posts as uncivil. The model flagged 21. But the model's 21 are mostly profanity-containing entertainment content, not necessarily what students perceive as uncivil. These are two different signals:

- Q8 measures perceived incivility ("did this feel rude or hostile")
- Detoxify measures profanity overlap with toxic-comment training data

These don't map cleanly to each other.

## Bottom line

Detoxify works for what it was built for (detecting toxic comments in formats like Wikipedia edits or Reddit hate speech) but it's a poor fit for social media entertainment content where profanity is common but not actually harmful. The 21 flagged posts are mostly false positives in terms of "actually toxic" content.

This is useful methodological evidence. Two follow-ups worth exploring:
- A toxicity model trained on social media (like the Cardiff Twitter offensive classifier) might do better
- An LLM could likely distinguish "casual entertainment swearing" from "actual toxicity"